# Setting up environment

In [1]:
!pip install ipywidgets

                                              0.0/139.8 kB ? eta -:--:--
     --------------------------------------   133.1/139.8 kB ? eta -:--:--
     -------------------------------------- 139.8/139.8 kB 2.8 MB/s eta 0:00:00
                                              0.0/2.2 MB ? eta -:--:--
     --------                                 0.5/2.2 MB 10.0 MB/s eta 0:00:01
     ----------------                         0.9/2.2 MB 9.7 MB/s eta 0:00:01
     ------------------------                 1.4/2.2 MB 9.7 MB/s eta 0:00:01
     --------------------------------         1.8/2.2 MB 9.5 MB/s eta 0:00:01
     ---------------------------------------  2.2/2.2 MB 9.3 MB/s eta 0:00:01
     ---------------------------------------- 2.2/2.2 MB 7.8 MB/s eta 0:00:00
                                              0.0/216.6 kB ? eta -:--:--
     ------------------------------------  215.0/216.6 kB 13.7 MB/s eta 0:00:01
     -------------------------------------- 216.6/216.6 kB 4.4 MB/s eta 0:00:00


[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import ipywidgets as widgets
from IPython.display import display, clear_output

# Application

In [5]:
# Load model + vocab
embeddings = torch.load("../artifacts/skipgram_embeddings.pt")
with open("../artifacts/word_to_index.pkl", "rb") as f:
    word_to_idx = pickle.load(f)
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

vocab = sorted(list(word_to_idx.keys()))

def get_vec(word):
    return embeddings[word_to_idx[word]]

def analogy_plot(a, b, c, top_k=5):
    with_output.clear_output()
    if not all(w in word_to_idx for w in [a, b, c]):
        with with_output:
            print("One of the words is missing from vocabulary.")
        return

    result_vec = get_vec(a) - get_vec(b) + get_vec(c)
    sims = torch.nn.functional.cosine_similarity(result_vec.unsqueeze(0), embeddings)
    top_idxs = sims.topk(top_k + 3).indices

    result_words = [idx_to_word[idx.item()] for idx in top_idxs if idx_to_word[idx.item()] not in [a, b, c]][:top_k]

    # Collect vectors for plotting
    words = [a, b, c] + result_words
    vectors = [get_vec(w) for w in words]

    pca = PCA(n_components=2)
    reduced = pca.fit_transform(torch.stack(vectors).numpy())

    # Plot
    with with_output:
        print(f"\n{a} - {b} + {c} ≈")
        for w in result_words:
            print(f"    {w}")

        plt.figure(figsize=(8,6))
        for i, word in enumerate(words):
            x, y = reduced[i]
            color = "purple" if word == result_words[0] else "black"
            plt.scatter(x, y, c=color)
            plt.text(x+0.01, y+0.01, word, fontsize=9)
        plt.title(f"{a} - {b} + {c}")
        plt.grid(True)
        plt.show()

# Widgets
a_dropdown = widgets.Dropdown(options=vocab, value="king", description="Word A:")
b_dropdown = widgets.Dropdown(options=vocab, value="man", description="Word B:")
c_dropdown = widgets.Dropdown(options=vocab, value="woman", description="Word C:")

go_button = widgets.Button(description="Run Analogy")
with_output = widgets.Output()

def on_button_click(_):
    analogy_plot(a_dropdown.value, b_dropdown.value, c_dropdown.value)

go_button.on_click(on_button_click)

display(widgets.VBox([a_dropdown, b_dropdown, c_dropdown, go_button, with_output]))

FileNotFoundError: [Errno 2] No such file or directory: '../artifacts/skipgram_embeddings.pt'